# Explorer la nomenclature

Avant d'écrire la moindre règle de nettoyage, on **regarde** les données.

L'ordre ci-dessous n'est pas arbitraire : c'est la séquence que suit un
ingénieur devant un jeu de données qu'il ne connaît pas.

| # | Question | Pourquoi |
|---|---|---|
| 1 | Combien de lignes, combien de colonnes ? | savoir à quoi on a affaire |
| 2 | À quoi ressemble **un** enregistrement ? | comprendre ce qu'est une ligne |
| 3 | Quelles colonnes sont vides, et combien ? | une colonne vide à 90 % est inutilisable |
| 4 | Combien de valeurs distinctes par colonne ? | distingue un identifiant d'une catégorie |
| 5 | Quelles valeurs reviennent le plus ? | voir le gros du corpus |
| 6 | Quelles **formes** prennent les valeurs ? | la technique qui permet d'écrire une règle juste |
| 7 | Qu'est-ce qui cloche ? | les anomalies sont des résultats, pas des accidents |

Exécutez chaque cellule avec **Maj + Entrée**.

## 1. Charger et mesurer

`csv.DictReader` lit chaque ligne comme un dictionnaire : `ligne["inn"]` donne la DCI.

`encoding="utf-8"` est obligatoire, sinon les accents se cassent.

In [ ]:
import csv
import re
from collections import Counter

CHEMIN = "../data/reference/drugs_semaine1.csv"

with open(CHEMIN, encoding="utf-8") as f:
    lignes = list(csv.DictReader(f))

colonnes = list(lignes[0].keys())
print(f"{len(lignes)} lignes, {len(colonnes)} colonnes")
colonnes

## 2. Regarder un enregistrement

Une ligne = **une boîte enregistrée**, pas un médicament. Le même médicament
apparaît plusieurs fois : dosages différents, formes différentes, laboratoires
différents.

C'est toute la difficulté du projet, et elle est visible dès maintenant.

In [ ]:
for cle, valeur in lignes[0].items():
    print(f"{cle:<20} {valeur}")

## 3. Complétude : quelles colonnes sont vides ?

**La première chose à mesurer sur un jeu de données inconnu.**

Une colonne remplie à 12 % ne sert à rien, mais on ne peut pas le deviner en
regardant trois lignes : il faut compter.

In [ ]:
print(f"{'colonne':<22}{'rempli':>8}")
print("-" * 32)
for col in colonnes:
    remplis = sum(1 for l in lignes if (l[col] or "").strip())
    pct = 100 * remplis / len(lignes)
    alerte = "  <-- presque vide" if pct < 50 else ""
    print(f"{col:<22}{pct:>7.0f}%{alerte}")

## 4. Cardinalité : combien de valeurs distinctes ?

Ce chiffre dit **ce qu'est** une colonne, sans avoir à le deviner :

- autant de valeurs distinctes que de lignes → un **identifiant**
- quelques valeurs seulement → une **catégorie**
- entre les deux → un **attribut** qui se répète

In [ ]:
print(f"{'colonne':<22}{'distinctes':>12}   nature probable")
print("-" * 60)
for col in colonnes:
    n = len({(l[col] or "").strip() for l in lignes if (l[col] or "").strip()})
    if n == 0:
        nature = "vide"
    elif n > 0.95 * len(lignes):
        nature = "identifiant"
    elif n <= 10:
        nature = "categorie"
    else:
        nature = "attribut"
    print(f"{col:<22}{n:>12}   {nature}")

## 5. Distribution : qu'est-ce qui revient le plus ?

`Counter` compte les occurrences. `.most_common(n)` donne les n plus fréquentes.

C'est l'outil que vous utiliserez le plus souvent.

In [ ]:
for col in ["inn", "form", "lab_country", "status", "type_code"]:
    print(f"\n--- {col} ---")
    for valeur, n in Counter(l[col] for l in lignes).most_common(5):
        print(f"  {n:>4}  {(valeur or '(vide)')[:56]}")

## 6. Les formes : la technique qui change tout

La colonne `packaging_raw` a des centaines de valeurs distinctes. Impossible de
les lire une par une — et c'est pourtant là-dessus qu'il faut écrire une règle.

**L'astuce : remplacer chaque nombre par `N`.** Les centaines de valeurs se
regroupent alors en une dizaine de *formes*, qu'on peut classer par fréquence.

Écrire une règle sans avoir fait ça, c'est deviner.

In [ ]:
def forme(valeur):
    """Remplace les nombres par N pour ne garder que la structure."""
    v = re.sub(r"\d+", "N", (valeur or "").upper())
    return re.sub(r"\s+", " ", v).strip()

valeurs = [l["packaging_raw"] for l in lignes if l["packaging_raw"].strip()]
print(f"{len(valeurs)} valeurs, {len(set(valeurs))} distinctes")
print(f"... mais seulement {len({forme(v) for v in valeurs})} formes :\n")

for f, n in Counter(forme(v) for v in valeurs).most_common(12):
    print(f"  {n:>4}  {100*n/len(valeurs):>5.1f}%   {f[:46]}")

**À lire dans ce tableau :** la forme la plus fréquente couvre une bonne part du
corpus. C'est elle qu'une règle doit traiter en premier. Les suivantes se
décident une par une — certaines sont lisibles, d'autres réellement ambiguës.

Faites le même exercice sur `strength_raw` en changeant une seule ligne.

In [ ]:
valeurs_dosage = [l["strength_raw"] for l in lignes if l["strength_raw"].strip()]
for f, n in Counter(forme(v) for v in valeurs_dosage).most_common(10):
    print(f"  {n:>4}  {100*n/len(valeurs_dosage):>5.1f}%   {f[:46]}")

## 7. Chercher ce qui cloche

Une anomalie n'est pas un accident à cacher : **c'est un résultat**. Le rapport
attend des chiffres, et « nous avons trouvé 3 incohérences dans la source
officielle » en est un.

Trois contrôles simples, chacun fondé sur une chose qui *devrait* être vraie.

In [ ]:
# a) Le numero d'enregistrement devrait etre unique.
doublons = [n for n, c in Counter(l["registration_no"] for l in lignes).items() if c > 1]
print(f"a) numeros d'enregistrement en double : {len(doublons)}")
for n in doublons[:3]:
    print(f"     {n}")

# b) La date finale ne devrait pas preceder la date initiale.
incoherentes = [l for l in lignes
                if l["date_final"] and l["date_initial"]
                and l["date_final"] < l["date_initial"]]
print(f"\nb) dates incoherentes (fin avant debut) : {len(incoherentes)}")
for l in incoherentes[:3]:
    print(f"     {l['brand_name']} : {l['date_initial']} -> {l['date_final']}")

# c) Le laboratoire ne devrait jamais ressembler a une substance.
suspects = [l for l in lignes
            if l["lab_name"] and l["inn"]
            and l["lab_name"].upper().startswith(l["inn"].upper()[:10])]
print(f"\nc) laboratoires qui ressemblent a une DCI : {len(suspects)}")
for l in suspects[:3]:
    print(f"     {l['brand_name']} : labo = {l['lab_name']}")

## 8. Le vrai problème du projet, en une cellule

Combien de marques différentes pour une même molécule, au même dosage et à la
même forme ? Un pharmacien sait qu'elles sont interchangeables. Une machine voit
des chaînes de caractères sans rapport.

In [ ]:
groupes = Counter(
    (l["inn"], l["strength_raw"], l["form"].split()[0] if l["form"] else "")
    for l in lignes
)

print("Les groupes les plus nombreux :\n")
for (dci, dosage, forme_), n in groupes.most_common(5):
    print(f"  {n:>3} marques   {dci[:34]:<34} {dosage:<14} {forme_}")

interchangeables = sum(n for g, n in groupes.items() if n > 1)
print(f"\n{interchangeables} produits sur {len(lignes)} ont au moins un equivalent.")

## Ce qu'il faut retenir

1. **Mesurer avant d'écrire.** Complétude, cardinalité, distribution, formes —
   dans cet ordre, avant la première ligne de nettoyage.
2. **Les formes plutôt que les valeurs.** Remplacer les nombres par `N` transforme
   des centaines de valeurs illisibles en une dizaine de cas décidables.
3. **Une anomalie est un résultat.** Elle se compte, se cite, et se met dans le
   rapport — elle ne se corrige pas en silence.
4. **Ne jamais deviner.** Une valeur illisible reste illisible. Inventer un nombre
   est la seule faute irrattrapable dans ce projet.

### Et après

Ce notebook sert à **comprendre**. Ce qui est décidé ici part ensuite dans
`src/pharmeazor/`, sous forme de fonctions testées — parce qu'un notebook ne se
rejoue pas tout seul et ne prouve rien.